# maritime_SQL_agent

## 선박 운항 컴플라이언스 관리를 위한 Text-to-SQL 분석 에이전트

이 노트북은 한 번에 실행할 수 있는 단일 Colab notebook입니다. 위에서부터 실행하면 다음을 수행합니다.

1. Neon PostgreSQL에 `maritime_sql_agent` schema 생성
2. 5개 테이블과 seed data 생성
3. row count, COMMENT, FK, expected SQL 검증
4. `NLSQLTableQueryEngine` baseline 실행
5. custom prompt + few-shot 개선
6. `ObjectIndex` 기반 table retrieval 비교
7. Gradio ChatInterface 데모 UI 정의

비밀값은 notebook에 포함하지 않았습니다. Colab Secrets에 `NEON_DSN`, `OPENAI_API_KEY`를 등록한 뒤 실행합니다.

## Project Brief Summary

- **Domain**: 선박 운항 컴플라이언스 운영 데이터베이스
- **Target users**: Captain, Chief Officer, Chief Engineer, Superintendent, DPA, Compliance 담당자, PSC/FSC inspection 준비자
- **Core workflow**: Ship Profile, Certificate expiry, PSC Checklist, Regulation/Manual/System Doc matching, Crew Training expiry를 자연어로 조회
- **Tables**: `vessels`, `certificates`, `compliance_documents`, `checklist_items`, `crew_training_records`
- **Evaluation set**: Easy 3개, Medium 4개, Hard 3개, 총 10개 질문
- **Data note**: 실제 운영 데이터가 아니라 maritime 업무 패턴을 참고한 synthetic seed data입니다.

```mermaid
erDiagram
    vessels ||--o{ certificates : "has"
    vessels ||--o{ checklist_items : "has"
    vessels ||--o{ crew_training_records : "has"
    vessels }o--o{ compliance_documents : "matched_by_ship_rules"
```

## Table Design Summary

| table | purpose | key SQL evaluation role |
|---|---|---|
| `vessels` | Ship Profile 기준 엔티티 | IMO lookup, fleet grouping, document matching |
| `certificates` | 선박 certificate expiry 관리 | expired/expiring certificate query, risk score |
| `compliance_documents` | Regulation, Manual, Checklist, System Doc catalog | ship-specific document matching |
| `checklist_items` | PSC/FSC/Vetting/Audit 준비 항목 | open item, completion rate, action item |
| `crew_training_records` | STCW 및 직무 교육 certificate 이력 | training expiry, crew eligibility risk |

Notebook 기본 실행은 DB 상태를 먼저 확인하고, 비어 있거나 row count가 맞지 않으면 schema와 seed data를 자동 구성하는 흐름입니다.
기존 데이터를 강제로 다시 만들 때만 `FORCE_DB_RESET = True`로 변경합니다.

In [ ]:
# 필요한 패키지 설치함
# PostgreSQL 연결, 표 출력, Gradio UI, LlamaIndex Text-to-SQL 구성에 필요함
%pip install -q sqlalchemy psycopg2-binary pandas tabulate gradio llama-index llama-index-llms-openai llama-index-embeddings-openai

In [ ]:
# 기본 라이브러리 불러옴
import os
import re
import textwrap
from typing import Iterable

# 데이터프레임과 DB 연결 도구 불러옴
import pandas as pd
from sqlalchemy import create_engine, event, text


# Colab Secrets 또는 환경변수에서 비밀값 읽음
# notebook 출력에 key나 DSN 찍지 않음
def load_secret(key: str, required: bool = True):
    value = os.getenv(key)
    if value:
        return value
    try:
        from google.colab import userdata
        value = userdata.get(key)
    except Exception:
        value = None
    if value:
        os.environ[key] = str(value)
        return str(value)
    if required:
        raise RuntimeError(f"{key} is not set. Register it in Colab Secrets.")
    return None


# 필수 비밀값 로드함
load_secret("NEON_DSN")
load_secret("OPENAI_API_KEY")
load_secret("LANGSMITH_API_KEY", required=False)

# LangSmith key가 있으면 tracing 활성화함
if os.getenv("LANGSMITH_API_KEY"):
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_PROJECT", "maritime_SQL_agent")

# 이 프로젝트의 schema와 5개 테이블로 고정
MARITIME_SCHEMA = "maritime_sql_agent"
MARITIME_TABLES = [
    "vessels",
    "certificates",
    "compliance_documents",
    "checklist_items",
    "crew_training_records",
]

# Neon PostgreSQL 연결 생성함
engine = create_engine(os.environ["NEON_DSN"])


# DB 연결마다 maritime_sql_agent schema를 기본 검색 경로로 설정함
@event.listens_for(engine, "connect")
def set_search_path(dbapi_connection, connection_record):
    cursor = dbapi_connection.cursor()
    cursor.execute(f"SET search_path TO {MARITIME_SCHEMA}, public")
    cursor.close()

print("Environment ready.")

In [ ]:
# notebook 내부에서 실행할 SQL script를 문자열로 보관함
# 제출 파일 하나만으로 schema, seed, 검증 SQL 확인 가능하게 함
SCHEMA_SQL = "CREATE SCHEMA IF NOT EXISTS maritime_sql_agent;\n\nSET search_path TO maritime_sql_agent, public;\n\nDROP TABLE IF EXISTS crew_training_records CASCADE;\nDROP TABLE IF EXISTS checklist_items CASCADE;\nDROP TABLE IF EXISTS certificates CASCADE;\nDROP TABLE IF EXISTS compliance_documents CASCADE;\nDROP TABLE IF EXISTS vessels CASCADE;\n\nCREATE TABLE vessels (\n    vessel_id        SERIAL PRIMARY KEY,\n    vessel_name      VARCHAR(120) NOT NULL,\n    imo_number       VARCHAR(20) NOT NULL UNIQUE,\n    vessel_type      VARCHAR(80) NOT NULL,\n    gross_tonnage    INT NOT NULL CHECK (gross_tonnage > 0),\n    build_year       INT NOT NULL CHECK (build_year >= 1900),\n    flag_state       VARCHAR(80) NOT NULL,\n    class_society    VARCHAR(40),\n    operating_status VARCHAR(20) NOT NULL CHECK (operating_status IN ('active', 'drydock', 'inactive'))\n);\n\nCOMMENT ON TABLE vessels IS '관리 대상 선박의 기본 제원과 운항 상태';\nCOMMENT ON COLUMN vessels.vessel_id IS '선박 고유 식별자';\nCOMMENT ON COLUMN vessels.vessel_name IS '선박명';\nCOMMENT ON COLUMN vessels.imo_number IS 'IMO Number';\nCOMMENT ON COLUMN vessels.vessel_type IS '선종: LNG Carrier, Bulk Carrier, Oil Tanker 등';\nCOMMENT ON COLUMN vessels.gross_tonnage IS '총톤수 GT';\nCOMMENT ON COLUMN vessels.build_year IS '선박 건조 연도';\nCOMMENT ON COLUMN vessels.flag_state IS '기국';\nCOMMENT ON COLUMN vessels.class_society IS '선급: KR, DNV, ABS, LR 등';\nCOMMENT ON COLUMN vessels.operating_status IS '운항 상태: active, drydock, inactive';\n\nCREATE TABLE certificates (\n    certificate_id     SERIAL PRIMARY KEY,\n    vessel_id          INT NOT NULL REFERENCES vessels(vessel_id) ON DELETE CASCADE,\n    certificate_name   VARCHAR(160) NOT NULL,\n    certificate_number VARCHAR(80),\n    issuing_authority  VARCHAR(120),\n    issue_date         DATE,\n    expiry_date        DATE NOT NULL,\n    document_url       TEXT,\n    remarks            TEXT\n);\n\nCOMMENT ON TABLE certificates IS '선박별 certificate와 만료일 관리 정보';\nCOMMENT ON COLUMN certificates.certificate_id IS 'Certificate 고유 식별자';\nCOMMENT ON COLUMN certificates.vessel_id IS 'Certificate가 연결된 선박 ID';\nCOMMENT ON COLUMN certificates.certificate_name IS 'Certificate 이름';\nCOMMENT ON COLUMN certificates.certificate_number IS 'Certificate 번호';\nCOMMENT ON COLUMN certificates.issuing_authority IS '발급 기관 또는 선급';\nCOMMENT ON COLUMN certificates.issue_date IS '발급일';\nCOMMENT ON COLUMN certificates.expiry_date IS '만료일';\nCOMMENT ON COLUMN certificates.document_url IS '스캔 PDF 또는 전자 문서 링크';\nCOMMENT ON COLUMN certificates.remarks IS '비고 또는 OCR 확인 메모';\n\nCREATE TABLE compliance_documents (\n    document_id         SERIAL PRIMARY KEY,\n    document_code       VARCHAR(80) NOT NULL UNIQUE,\n    document_title      VARCHAR(220) NOT NULL,\n    document_type       VARCHAR(40) NOT NULL CHECK (document_type IN ('Regulation', 'Manual', 'Checklist', 'System Doc', 'Manufacturer Manual')),\n    document_domain     VARCHAR(50) NOT NULL,\n    applies_vessel_type VARCHAR(80),\n    tonnage_min         INT CHECK (tonnage_min IS NULL OR tonnage_min >= 0),\n    tonnage_max         INT CHECK (tonnage_max IS NULL OR tonnage_max >= 0),\n    build_year_min      INT CHECK (build_year_min IS NULL OR build_year_min >= 1900),\n    build_year_max      INT CHECK (build_year_max IS NULL OR build_year_max >= 1900),\n    issuer              VARCHAR(120),\n    is_mandatory        BOOLEAN NOT NULL DEFAULT TRUE,\n    CHECK (tonnage_min IS NULL OR tonnage_max IS NULL OR tonnage_min <= tonnage_max),\n    CHECK (build_year_min IS NULL OR build_year_max IS NULL OR build_year_min <= build_year_max)\n);\n\nCOMMENT ON TABLE compliance_documents IS '선박별 적용 가능성을 판단할 Regulation, Manual, Checklist, System Doc 목록';\nCOMMENT ON COLUMN compliance_documents.document_id IS '근거 문서 고유 식별자';\nCOMMENT ON COLUMN compliance_documents.document_code IS '문서 코드 또는 약어';\nCOMMENT ON COLUMN compliance_documents.document_title IS '문서 제목';\nCOMMENT ON COLUMN compliance_documents.document_type IS '문서 유형: Regulation, Manual, Checklist, System Doc, Manufacturer Manual';\nCOMMENT ON COLUMN compliance_documents.document_domain IS '문서 업무 영역: Safety, Navigation, Cargo, Environment 등';\nCOMMENT ON COLUMN compliance_documents.applies_vessel_type IS '적용 선종. NULL이면 모든 선종에 적용';\nCOMMENT ON COLUMN compliance_documents.tonnage_min IS '적용 최소 GT. NULL이면 하한 없음';\nCOMMENT ON COLUMN compliance_documents.tonnage_max IS '적용 최대 GT. NULL이면 상한 없음';\nCOMMENT ON COLUMN compliance_documents.build_year_min IS '적용 최소 건조 연도. NULL이면 하한 없음';\nCOMMENT ON COLUMN compliance_documents.build_year_max IS '적용 최대 건조 연도. NULL이면 상한 없음';\nCOMMENT ON COLUMN compliance_documents.issuer IS '발행 기관, 회사, 제조사';\nCOMMENT ON COLUMN compliance_documents.is_mandatory IS '필수 적용 문서 여부';\n\nCREATE TABLE checklist_items (\n    item_id        SERIAL PRIMARY KEY,\n    vessel_id      INT NOT NULL REFERENCES vessels(vessel_id) ON DELETE CASCADE,\n    checklist_type VARCHAR(30) NOT NULL CHECK (checklist_type IN ('PSC', 'FSC', 'Vetting', 'Audit')),\n    item_category  VARCHAR(60) NOT NULL,\n    item_text      TEXT NOT NULL,\n    status         VARCHAR(20) NOT NULL CHECK (status IN ('open', 'in_progress', 'completed', 'deferred')),\n    severity       VARCHAR(20) NOT NULL CHECK (severity IN ('low', 'medium', 'high', 'critical')),\n    due_date       DATE,\n    completed_at   DATE,\n    remarks        TEXT,\n    CHECK (completed_at IS NULL OR due_date IS NULL OR completed_at >= due_date - INTERVAL '365 days')\n);\n\nCOMMENT ON TABLE checklist_items IS 'PSC, FSC, Vetting, Audit 준비를 위한 선박별 체크리스트 항목';\nCOMMENT ON COLUMN checklist_items.item_id IS '체크리스트 항목 고유 식별자';\nCOMMENT ON COLUMN checklist_items.vessel_id IS '체크리스트가 연결된 선박 ID';\nCOMMENT ON COLUMN checklist_items.checklist_type IS '점검 유형: PSC, FSC, Vetting, Audit';\nCOMMENT ON COLUMN checklist_items.item_category IS '점검 카테고리: Bridge, Deck, Engine, Safety 등';\nCOMMENT ON COLUMN checklist_items.item_text IS '점검 항목 내용';\nCOMMENT ON COLUMN checklist_items.status IS '항목 처리 상태';\nCOMMENT ON COLUMN checklist_items.severity IS '미완료 시 중요도';\nCOMMENT ON COLUMN checklist_items.due_date IS '조치 기한';\nCOMMENT ON COLUMN checklist_items.completed_at IS '완료일';\nCOMMENT ON COLUMN checklist_items.remarks IS '점검 비고';\n\nCREATE TABLE crew_training_records (\n    training_id        SERIAL PRIMARY KEY,\n    vessel_id          INT NOT NULL REFERENCES vessels(vessel_id) ON DELETE CASCADE,\n    crew_name          VARCHAR(120) NOT NULL,\n    rank               VARCHAR(80) NOT NULL,\n    training_name      VARCHAR(160) NOT NULL,\n    certificate_number VARCHAR(80),\n    issuing_authority  VARCHAR(120),\n    issue_date         DATE NOT NULL,\n    expiry_date        DATE,\n    is_required        BOOLEAN NOT NULL DEFAULT TRUE,\n    CHECK (expiry_date IS NULL OR expiry_date >= issue_date)\n);\n\nCOMMENT ON TABLE crew_training_records IS '선원별 STCW 및 직무 교육 certificate 이력';\nCOMMENT ON COLUMN crew_training_records.training_id IS '교육 이력 고유 식별자';\nCOMMENT ON COLUMN crew_training_records.vessel_id IS '선원이 배정된 선박 ID';\nCOMMENT ON COLUMN crew_training_records.crew_name IS '선원 이름';\nCOMMENT ON COLUMN crew_training_records.rank IS '직급: Captain, Chief Officer, 2/O, Chief Engineer 등';\nCOMMENT ON COLUMN crew_training_records.training_name IS '교육 또는 certificate 이름';\nCOMMENT ON COLUMN crew_training_records.certificate_number IS '교육 certificate 번호';\nCOMMENT ON COLUMN crew_training_records.issuing_authority IS '교육 발급 기관';\nCOMMENT ON COLUMN crew_training_records.issue_date IS '교육 certificate 발급일';\nCOMMENT ON COLUMN crew_training_records.expiry_date IS '교육 certificate 만료일. NULL이면 만료 없음';\nCOMMENT ON COLUMN crew_training_records.is_required IS '승선 또는 직무 수행에 필수인 교육 여부';\n\nCREATE INDEX idx_certificates_vessel_expiry\n    ON certificates (vessel_id, expiry_date);\n\nCREATE INDEX idx_checklist_vessel_type_status\n    ON checklist_items (vessel_id, checklist_type, status, severity);\n\nCREATE INDEX idx_training_vessel_expiry\n    ON crew_training_records (vessel_id, expiry_date);\n\nCREATE INDEX idx_documents_matching_rules\n    ON compliance_documents (applies_vessel_type, tonnage_min, tonnage_max, build_year_min, build_year_max);\n\n"

SEED_SQL = "SET search_path TO maritime_sql_agent, public;\n\nTRUNCATE TABLE\n    crew_training_records,\n    checklist_items,\n    certificates,\n    compliance_documents,\n    vessels\nRESTART IDENTITY CASCADE;\n\nINSERT INTO vessels (\n    vessel_id, vessel_name, imo_number, vessel_type, gross_tonnage,\n    build_year, flag_state, class_society, operating_status\n) VALUES\n    (1, 'M/V Ocean Aurora', '9876543', 'LNG Carrier', 116000, 2018, 'Marshall Islands', 'DNV', 'active'),\n    (2, 'M/V Pacific Trader', '9765432', 'Bulk Carrier', 45200, 2012, 'Panama', 'KR', 'active'),\n    (3, 'M/V Hanseatic Star', '9654321', 'Container Ship', 74000, 2020, 'Liberia', 'ABS', 'active'),\n    (4, 'M/T Blue Horizon', '9543210', 'Oil Tanker', 58000, 2015, 'Singapore', 'LR', 'active'),\n    (5, 'M/V Coastal Hope', '9432109', 'General Cargo', 8500, 2008, 'Republic of Korea', 'KR', 'drydock');\n\nINSERT INTO vessels (\n    vessel_id, vessel_name, imo_number, vessel_type, gross_tonnage,\n    build_year, flag_state, class_society, operating_status\n)\nSELECT\n    gs AS vessel_id,\n    'M/V Training Vessel ' || LPAD((gs - 5)::TEXT, 2, '0') AS vessel_name,\n    (9300000 + gs)::TEXT AS imo_number,\n    CASE gs % 5\n        WHEN 0 THEN 'General Cargo'\n        WHEN 1 THEN 'Bulk Carrier'\n        WHEN 2 THEN 'Container Ship'\n        WHEN 3 THEN 'Offshore Support Vessel'\n        ELSE 'Training Vessel'\n    END AS vessel_type,\n    9000 + (gs * 850) AS gross_tonnage,\n    2000 + (gs % 24) AS build_year,\n    CASE gs % 5\n        WHEN 0 THEN 'Panama'\n        WHEN 1 THEN 'Liberia'\n        WHEN 2 THEN 'Marshall Islands'\n        WHEN 3 THEN 'Singapore'\n        ELSE 'Republic of Korea'\n    END AS flag_state,\n    CASE gs % 4\n        WHEN 0 THEN 'KR'\n        WHEN 1 THEN 'DNV'\n        WHEN 2 THEN 'ABS'\n        ELSE 'LR'\n    END AS class_society,\n    CASE\n        WHEN gs % 11 = 0 THEN 'drydock'\n        WHEN gs % 17 = 0 THEN 'inactive'\n        ELSE 'active'\n    END AS operating_status\nFROM generate_series(6, 50) AS gs;\n\nSELECT setval(pg_get_serial_sequence('vessels', 'vessel_id'), (SELECT MAX(vessel_id) FROM vessels));\n\nINSERT INTO certificates (\n    vessel_id, certificate_name, certificate_number, issuing_authority,\n    issue_date, expiry_date, document_url, remarks\n) VALUES\n    (1, 'Cargo Ship Safety Equipment Certificate', 'SE-2025-001', 'DNV', '2025-05-26', '2026-05-25', 'https://docs.example.com/cert/se-2025-001.pdf', 'renewal survey booked'),\n    (1, 'International Oil Pollution Prevention Certificate', 'IOPP-2025-011', 'DNV', '2025-01-15', '2027-01-15', 'https://docs.example.com/cert/iopp-2025-011.pdf', 'valid'),\n    (2, 'Cargo Ship Safety Construction Certificate', 'SC-2021-009', 'KR', '2021-04-20', '2026-04-20', 'https://docs.example.com/cert/sc-2021-009.pdf', 'expired, urgent renewal'),\n    (2, 'International Load Line Certificate', 'LL-2021-009', 'KR', '2021-07-15', '2026-07-15', 'https://docs.example.com/cert/ll-2021-009.pdf', 'renewal window open'),\n    (3, 'Cargo Ship Safety Radio Certificate', 'SR-2025-077', 'ABS', '2025-06-21', '2026-06-20', 'https://docs.example.com/cert/sr-2025-077.pdf', 'GMDSS inspection required'),\n    (3, 'International Tonnage Certificate', 'ITC-2020-014', 'ABS', '2020-03-01', '2030-03-01', 'https://docs.example.com/cert/itc-2020-014.pdf', 'valid'),\n    (4, 'International Oil Pollution Prevention Certificate', 'IOPP-2021-041', 'LR', '2021-05-30', '2026-05-30', 'https://docs.example.com/cert/iopp-2021-041.pdf', 'annual survey pending'),\n    (4, 'Classification Certificate', 'CLASS-2021-041', 'LR', '2021-10-01', '2026-10-01', 'https://docs.example.com/cert/class-2021-041.pdf', 'valid'),\n    (5, 'Cargo Ship Safety Equipment Certificate', 'SE-2021-088', 'KR', '2021-05-10', '2026-05-10', 'https://docs.example.com/cert/se-2021-088.pdf', 'drydock renewal planned'),\n    (5, 'Certificate of Registry', 'REG-2023-088', 'Korea Register', '2023-01-01', '2028-01-01', 'https://docs.example.com/cert/reg-2023-088.pdf', 'valid');\n\nINSERT INTO certificates (\n    vessel_id, certificate_name, certificate_number, issuing_authority,\n    issue_date, expiry_date, document_url, remarks\n)\nSELECT\n    v.vessel_id,\n    cert_def.certificate_name,\n    cert_def.prefix || '-' || v.vessel_id::TEXT || '-' || cert_def.seq::TEXT AS certificate_number,\n    v.class_society,\n    DATE '2025-01-01' + ((v.vessel_id + cert_def.seq) % 180) * INTERVAL '1 day' AS issue_date,\n    DATE '2027-07-01' + ((v.vessel_id + cert_def.seq) % 365) * INTERVAL '1 day' AS expiry_date,\n    'https://docs.example.com/cert/generated-' || v.vessel_id::TEXT || '-' || cert_def.seq::TEXT || '.pdf' AS document_url,\n    'generated valid certificate for seed dataset' AS remarks\nFROM vessels v\nCROSS JOIN (\n    VALUES\n        (1, 'Cargo Ship Safety Certificate', 'CSS'),\n        (2, 'Maritime Labour Certificate', 'MLC')\n) AS cert_def(seq, certificate_name, prefix)\nWHERE v.vessel_id BETWEEN 6 AND 50;\n\nINSERT INTO compliance_documents (\n    document_code, document_title, document_type, document_domain,\n    applies_vessel_type, tonnage_min, tonnage_max, build_year_min, build_year_max,\n    issuer, is_mandatory\n) VALUES\n    ('SOLAS-III', 'SOLAS Chapter III Life-Saving Appliances', 'Regulation', 'Safety', NULL, NULL, NULL, NULL, NULL, 'IMO', TRUE),\n    ('SOLAS-V', 'SOLAS Chapter V Safety of Navigation', 'Regulation', 'Navigation', NULL, NULL, NULL, NULL, NULL, 'IMO', TRUE),\n    ('MARPOL-I', 'MARPOL Annex I Oil Pollution Prevention', 'Regulation', 'Environment', 'Oil Tanker', NULL, NULL, NULL, NULL, 'IMO', TRUE),\n    ('MARPOL-I-LNG', 'MARPOL Annex I Machinery Space Bilge Requirements', 'Regulation', 'Environment', 'LNG Carrier', NULL, NULL, NULL, NULL, 'IMO', TRUE),\n    ('ISM-CODE', 'International Safety Management Code', 'Regulation', 'Management', NULL, NULL, NULL, NULL, NULL, 'IMO', TRUE),\n    ('STCW-A-II', 'STCW Deck Officer Competence Standards', 'Regulation', 'Crew', NULL, NULL, NULL, NULL, NULL, 'IMO', TRUE),\n    ('PSC-GENERAL', 'PSC Pre-Inspection Checklist', 'Checklist', 'Inspection', NULL, 500, NULL, NULL, NULL, 'Shipping Company', TRUE),\n    ('LNG-CARGO-MANUAL', 'LNG Carrier Cargo Operation Manual', 'Manual', 'Cargo', 'LNG Carrier', NULL, NULL, NULL, NULL, 'Shipping Company', TRUE),\n    ('CARGO-SYSTEM-DOC', 'LNG Cargo Handling System Doc', 'System Doc', 'Cargo', 'LNG Carrier', NULL, NULL, 2010, NULL, 'Shipyard', FALSE),\n    ('PMS-ENGINE-MANUAL', 'Engine PMS Manufacturer Manual', 'Manufacturer Manual', 'Engineering', NULL, NULL, NULL, NULL, NULL, 'Equipment Maker', FALSE);\n\nINSERT INTO compliance_documents (\n    document_code, document_title, document_type, document_domain,\n    applies_vessel_type, tonnage_min, tonnage_max, build_year_min, build_year_max,\n    issuer, is_mandatory\n)\nSELECT\n    'DOC-' || LPAD(gs::TEXT, 3, '0') AS document_code,\n    'Generated Fleet Procedure ' || gs::TEXT AS document_title,\n    CASE gs % 5\n        WHEN 0 THEN 'Regulation'\n        WHEN 1 THEN 'Manual'\n        WHEN 2 THEN 'Checklist'\n        WHEN 3 THEN 'System Doc'\n        ELSE 'Manufacturer Manual'\n    END AS document_type,\n    CASE gs % 6\n        WHEN 0 THEN 'Safety'\n        WHEN 1 THEN 'Navigation'\n        WHEN 2 THEN 'Cargo'\n        WHEN 3 THEN 'Environment'\n        WHEN 4 THEN 'Engineering'\n        ELSE 'Inspection'\n    END AS document_domain,\n    CASE gs % 4\n        WHEN 0 THEN 'Training Vessel'\n        WHEN 1 THEN 'Offshore Support Vessel'\n        WHEN 2 THEN 'General Cargo'\n        ELSE 'Container Ship'\n    END AS applies_vessel_type,\n    CASE WHEN gs % 3 = 0 THEN 5000 ELSE NULL END AS tonnage_min,\n    CASE WHEN gs % 7 = 0 THEN 80000 ELSE NULL END AS tonnage_max,\n    CASE WHEN gs % 4 = 0 THEN 2005 ELSE NULL END AS build_year_min,\n    NULL AS build_year_max,\n    CASE gs % 3\n        WHEN 0 THEN 'Shipping Company'\n        WHEN 1 THEN 'Class Society'\n        ELSE 'Equipment Maker'\n    END AS issuer,\n    (gs % 5 <> 0) AS is_mandatory\nFROM generate_series(11, 50) AS gs;\n\nINSERT INTO checklist_items (\n    vessel_id, checklist_type, item_category, item_text,\n    status, severity, due_date, completed_at, remarks\n) VALUES\n    (1, 'PSC', 'Safety', 'Lifeboat weekly test record verified', 'completed', 'high', '2026-05-01', '2026-05-01', 'evidence uploaded'),\n    (1, 'PSC', 'Cargo', 'Cargo ESD trip test evidence prepared', 'open', 'high', '2026-05-08', NULL, 'pending cargo team confirmation'),\n    (1, 'PSC', 'Safety', 'Fire door inspection log updated', 'in_progress', 'medium', '2026-05-06', NULL, 'engine team updating log'),\n    (2, 'PSC', 'Deck', 'Hatch cover condition verified', 'open', 'critical', '2026-05-05', NULL, 'repair estimate requested'),\n    (2, 'PSC', 'Navigation', 'Charts and publications corrected', 'completed', 'medium', '2026-05-02', '2026-05-02', 'checked by Chief Officer'),\n    (3, 'PSC', 'Radio', 'GMDSS logbook checked', 'completed', 'medium', '2026-05-02', '2026-05-02', 'radio officer confirmed'),\n    (3, 'PSC', 'Safety', 'Fire drill record signed by Master', 'open', 'high', '2026-05-12', NULL, 'Master signature pending'),\n    (4, 'PSC', 'Environment', 'SOPEP drill evidence prepared', 'open', 'medium', '2026-05-07', NULL, 'drill photo pending'),\n    (4, 'PSC', 'Environment', 'Oil Record Book entries reviewed', 'completed', 'high', '2026-05-01', '2026-05-01', 'reviewed by Chief Engineer'),\n    (5, 'PSC', 'Safety', 'Muster list updated after crew change', 'open', 'high', '2026-05-03', NULL, 'update required before undocking');\n\nINSERT INTO checklist_items (\n    vessel_id, checklist_type, item_category, item_text,\n    status, severity, due_date, completed_at, remarks\n)\nSELECT\n    v.vessel_id,\n    'PSC' AS checklist_type,\n    CASE v.vessel_id % 5\n        WHEN 0 THEN 'Documentation'\n        WHEN 1 THEN 'Bridge'\n        WHEN 2 THEN 'Lifesaving'\n        WHEN 3 THEN 'Fire Safety'\n        ELSE 'Pollution'\n    END AS item_category,\n    CASE v.vessel_id % 15\n        WHEN 0 THEN 'Certificates validity checked'\n        WHEN 1 THEN 'Crew lists and licenses verified'\n        WHEN 2 THEN 'Oil Record Book updated'\n        WHEN 3 THEN 'Garbage Record Book updated'\n        WHEN 4 THEN 'Hours of Work and Rest records reviewed'\n        WHEN 5 THEN 'Nautical publications updated with latest NtM'\n        WHEN 6 THEN 'Charts corrected for intended voyage'\n        WHEN 7 THEN 'Voyage plan prepared berth-to-berth'\n        WHEN 8 THEN 'GMDSS daily and weekly test records available'\n        WHEN 9 THEN 'Steering gear test record verified'\n        WHEN 10 THEN 'Lifeboat engine start test completed within three minutes'\n        WHEN 11 THEN 'Lifebuoys lights and smoke signals checked'\n        WHEN 12 THEN 'Emergency Fire Pump pressure test completed'\n        WHEN 13 THEN 'Fire detection panel checked with no active faults'\n        ELSE 'OWS 15ppm alarm test evidence available'\n    END AS item_text,\n    'completed' AS status,\n    CASE v.vessel_id % 3\n        WHEN 0 THEN 'low'\n        WHEN 1 THEN 'medium'\n        ELSE 'low'\n    END AS severity,\n    DATE '2026-04-01' + (v.vessel_id % 30) * INTERVAL '1 day' AS due_date,\n    DATE '2026-04-02' + (v.vessel_id % 30) * INTERVAL '1 day' AS completed_at,\n    'PSC template item adapted from maritime operations reference pattern' AS remarks\nFROM vessels v\nWHERE v.vessel_id BETWEEN 6 AND 50;\n\nINSERT INTO crew_training_records (\n    vessel_id, crew_name, rank, training_name, certificate_number,\n    issuing_authority, issue_date, expiry_date, is_required\n) VALUES\n    (1, 'Kim Minjun', 'Captain', 'Advanced Fire Fighting', 'AFF-2024-001', 'Korea Maritime Training Center', '2024-02-01', '2027-02-01', TRUE),\n    (1, 'Maria Santos', 'Chief Officer', 'Advanced Cargo Operations', 'ACO-2021-014', 'SIGTTO Training Center', '2021-05-20', '2026-05-20', TRUE),\n    (1, 'Lee Jisoo', '2/O', 'ECDIS Generic Training', 'ECDIS-2021-019', 'KR Academy', '2021-04-30', '2026-04-30', TRUE),\n    (2, 'Ahmed Khan', 'Chief Engineer', 'Basic Safety Training', 'BST-2021-055', 'Maritime Academy', '2021-09-01', '2026-09-01', TRUE),\n    (2, 'Ravi Singh', 'Bosun', 'Enclosed Space Entry', 'ESE-2021-078', 'Maritime Academy', '2021-05-15', '2026-05-15', TRUE),\n    (3, 'Zhang Wei', '2/O', 'GMDSS GOC', 'GMDSS-2021-120', 'Radio School', '2021-06-10', '2026-06-10', TRUE),\n    (3, 'Jose Santos', 'AB', 'Basic Safety Training', 'BST-2020-099', 'Maritime Academy', '2020-12-31', '2025-12-31', TRUE),\n    (4, 'Park Hyun', 'Chief Officer', 'Oil Tanker Advanced Training', 'OTA-2021-041', 'Tanker Training Center', '2021-05-28', '2026-05-28', TRUE),\n    (4, 'Chen Li', '3/E', 'Advanced Fire Fighting', 'AFF-2025-022', 'Maritime Academy', '2025-03-01', '2028-03-01', TRUE),\n    (5, 'Choi Ara', 'Master', 'Medical Care', 'MED-2021-033', 'Maritime Medical Institute', '2021-05-05', '2026-05-05', TRUE);\n\nINSERT INTO crew_training_records (\n    vessel_id, crew_name, rank, training_name, certificate_number,\n    issuing_authority, issue_date, expiry_date, is_required\n)\nSELECT\n    v.vessel_id,\n    'Crew Member ' || LPAD(v.vessel_id::TEXT, 3, '0') AS crew_name,\n    CASE v.vessel_id % 6\n        WHEN 0 THEN 'Captain'\n        WHEN 1 THEN 'Chief Officer'\n        WHEN 2 THEN 'Chief Engineer'\n        WHEN 3 THEN '2/O'\n        WHEN 4 THEN '3/E'\n        ELSE 'AB'\n    END AS rank,\n    CASE v.vessel_id % 5\n        WHEN 0 THEN 'Basic Safety Training'\n        WHEN 1 THEN 'Advanced Fire Fighting'\n        WHEN 2 THEN 'ECDIS Generic Training'\n        WHEN 3 THEN 'Medical First Aid'\n        ELSE 'Bridge Resource Management'\n    END AS training_name,\n    'TRN-' || v.vessel_id::TEXT || '-001' AS certificate_number,\n    CASE v.vessel_id % 3\n        WHEN 0 THEN 'Maritime Academy'\n        WHEN 1 THEN 'KR Academy'\n        ELSE 'Korea Maritime Training Center'\n    END AS issuing_authority,\n    DATE '2025-01-01' + (v.vessel_id % 120) * INTERVAL '1 day' AS issue_date,\n    DATE '2027-01-01' + (v.vessel_id % 240) * INTERVAL '1 day' AS expiry_date,\n    TRUE AS is_required\nFROM vessels v\nWHERE v.vessel_id BETWEEN 6 AND 50;\n"

VERIFY_SQL = "SET search_path TO maritime_sql_agent, public;\n\n-- 1. Row count check: every table should have at least 50 rows.\nSELECT 'vessels' AS table_name, COUNT(*) AS row_count FROM vessels\nUNION ALL\nSELECT 'certificates', COUNT(*) FROM certificates\nUNION ALL\nSELECT 'compliance_documents', COUNT(*) FROM compliance_documents\nUNION ALL\nSELECT 'checklist_items', COUNT(*) FROM checklist_items\nUNION ALL\nSELECT 'crew_training_records', COUNT(*) FROM crew_training_records\nORDER BY table_name;\n\n-- 2. Column comment check: missing_comment_count should be 0.\nSELECT\n    c.table_name,\n    COUNT(*) FILTER (\n        WHERE pg_catalog.col_description(format('%I.%I', c.table_schema, c.table_name)::REGCLASS::OID, c.ordinal_position) IS NULL\n    ) AS missing_comment_count\nFROM information_schema.columns c\nWHERE c.table_schema = 'maritime_sql_agent'\n  AND c.table_name IN ('vessels', 'certificates', 'compliance_documents', 'checklist_items', 'crew_training_records')\nGROUP BY c.table_name\nORDER BY c.table_name;\n\n-- 3. FK orphan check: every count should be 0.\nSELECT 'certificates_without_vessel' AS check_name, COUNT(*) AS invalid_count\nFROM certificates c\nLEFT JOIN vessels v ON v.vessel_id = c.vessel_id\nWHERE v.vessel_id IS NULL\nUNION ALL\nSELECT 'checklist_items_without_vessel', COUNT(*)\nFROM checklist_items ci\nLEFT JOIN vessels v ON v.vessel_id = ci.vessel_id\nWHERE v.vessel_id IS NULL\nUNION ALL\nSELECT 'crew_training_without_vessel', COUNT(*)\nFROM crew_training_records ctr\nLEFT JOIN vessels v ON v.vessel_id = ctr.vessel_id\nWHERE v.vessel_id IS NULL;\n\n-- 4. Key expected rows for the 5-vessel briefing sample.\nSELECT\n    'sample_expiring_certificates_q2' AS check_name,\n    COUNT(*) AS expected_sample_count\nFROM certificates c\nJOIN vessels v ON v.vessel_id = c.vessel_id\nWHERE v.vessel_id BETWEEN 1 AND 5\n  AND c.expiry_date <= DATE '2026-05-02' + INTERVAL '60 days'\nUNION ALL\nSELECT\n    'sample_open_high_critical_psc_q3',\n    COUNT(*)\nFROM checklist_items ci\nJOIN vessels v ON v.vessel_id = ci.vessel_id\nWHERE v.vessel_id BETWEEN 1 AND 5\n  AND ci.checklist_type = 'PSC'\n  AND ci.status <> 'completed'\n  AND ci.severity IN ('high', 'critical')\nUNION ALL\nSELECT\n    'sample_required_training_at_risk_q9',\n    COUNT(*)\nFROM crew_training_records ctr\nJOIN vessels v ON v.vessel_id = ctr.vessel_id\nWHERE v.vessel_id BETWEEN 1 AND 5\n  AND ctr.is_required = TRUE\n  AND ctr.expiry_date <= DATE '2026-05-02' + INTERVAL '30 days';\n\n"

EXPECTED_SQL_SCRIPT = "SET search_path TO maritime_sql_agent, public;\n\n-- Q1 Easy: IMO 번호 9876543인 선박의 기본 제원\nSELECT\n    vessel_name,\n    vessel_type,\n    gross_tonnage,\n    build_year,\n    flag_state,\n    class_society,\n    operating_status\nFROM vessels\nWHERE imo_number = '9876543';\n\n-- Q2 Easy: 2026-05-02 기준 만료 또는 60일 이내 만료 certificate\nSELECT\n    v.vessel_name,\n    c.certificate_name,\n    c.expiry_date,\n    c.expiry_date - DATE '2026-05-02' AS days_remaining,\n    CASE\n        WHEN c.expiry_date < DATE '2026-05-02' THEN 'expired'\n        ELSE 'expiring'\n    END AS cert_status\nFROM certificates c\nJOIN vessels v ON v.vessel_id = c.vessel_id\nWHERE c.expiry_date <= DATE '2026-05-02' + INTERVAL '60 days'\nORDER BY c.expiry_date;\n\n-- Q3 Easy: 미완료 high/critical PSC checklist item\nSELECT\n    v.vessel_name,\n    ci.item_category,\n    ci.item_text,\n    ci.severity,\n    ci.due_date\nFROM checklist_items ci\nJOIN vessels v ON v.vessel_id = ci.vessel_id\nWHERE ci.checklist_type = 'PSC'\n  AND ci.status <> 'completed'\n  AND ci.severity IN ('high', 'critical')\nORDER BY\n    CASE ci.severity WHEN 'critical' THEN 1 WHEN 'high' THEN 2 ELSE 3 END,\n    ci.due_date;\n\n-- Q4 Medium: 이미 만료된 required crew training certificate\nSELECT\n    v.vessel_name,\n    ctr.crew_name,\n    ctr.rank,\n    ctr.training_name,\n    ctr.expiry_date\nFROM crew_training_records ctr\nJOIN vessels v ON v.vessel_id = ctr.vessel_id\nWHERE ctr.is_required = TRUE\n  AND ctr.expiry_date < DATE '2026-05-02'\nORDER BY ctr.expiry_date;\n\n-- Q5 Medium: 선박별 expired/90일 이내 만료 certificate count\nSELECT\n    v.vessel_name,\n    COUNT(*) FILTER (WHERE c.expiry_date < DATE '2026-05-02') AS expired_count,\n    COUNT(*) FILTER (\n        WHERE c.expiry_date >= DATE '2026-05-02'\n          AND c.expiry_date <= DATE '2026-05-02' + INTERVAL '90 days'\n    ) AS expiring_90d_count\nFROM vessels v\nLEFT JOIN certificates c ON c.vessel_id = v.vessel_id\nGROUP BY v.vessel_id, v.vessel_name\nORDER BY expired_count DESC, expiring_90d_count DESC, v.vessel_name;\n\n-- Q6 Medium: 선박별 PSC checklist completion rate\nSELECT\n    v.vessel_name,\n    COUNT(ci.item_id) AS total_items,\n    COUNT(ci.item_id) FILTER (WHERE ci.status = 'completed') AS completed_items,\n    ROUND(\n        100.0 * COUNT(ci.item_id) FILTER (WHERE ci.status = 'completed')\n        / NULLIF(COUNT(ci.item_id), 0),\n        2\n    ) AS completion_pct\nFROM vessels v\nJOIN checklist_items ci ON ci.vessel_id = v.vessel_id\nWHERE ci.checklist_type = 'PSC'\nGROUP BY v.vessel_id, v.vessel_name\nORDER BY completion_pct ASC, v.vessel_name;\n\n-- Q7 Medium: M/V Ocean Aurora에 적용되는 Regulation, Manual, System Doc\nSELECT\n    v.vessel_name,\n    d.document_code,\n    d.document_title,\n    d.document_type,\n    d.document_domain,\n    d.is_mandatory\nFROM vessels v\nJOIN compliance_documents d\n    ON (d.applies_vessel_type IS NULL OR d.applies_vessel_type = v.vessel_type)\n   AND (d.tonnage_min IS NULL OR d.tonnage_min <= v.gross_tonnage)\n   AND (d.tonnage_max IS NULL OR d.tonnage_max >= v.gross_tonnage)\n   AND (d.build_year_min IS NULL OR d.build_year_min <= v.build_year)\n   AND (d.build_year_max IS NULL OR d.build_year_max >= v.build_year)\nWHERE v.vessel_name = 'M/V Ocean Aurora'\nORDER BY d.document_domain, d.document_type, d.document_code;\n\n-- Q8 Hard: fleet compliance risk ranking\nWITH cert_risk AS (\n    SELECT\n        vessel_id,\n        COUNT(*) FILTER (WHERE expiry_date < DATE '2026-05-02') AS expired_cert_count,\n        COUNT(*) FILTER (\n            WHERE expiry_date >= DATE '2026-05-02'\n              AND expiry_date <= DATE '2026-05-02' + INTERVAL '90 days'\n        ) AS expiring_cert_count\n    FROM certificates\n    GROUP BY vessel_id\n),\nchecklist_risk AS (\n    SELECT\n        vessel_id,\n        COUNT(*) FILTER (WHERE status <> 'completed' AND severity = 'critical') AS open_critical_count,\n        COUNT(*) FILTER (WHERE status <> 'completed' AND severity = 'high') AS open_high_count\n    FROM checklist_items\n    WHERE checklist_type = 'PSC'\n    GROUP BY vessel_id\n),\nscored AS (\n    SELECT\n        v.vessel_name,\n        COALESCE(cr.expired_cert_count, 0) AS expired_cert_count,\n        COALESCE(cr.expiring_cert_count, 0) AS expiring_cert_count,\n        COALESCE(ch.open_critical_count, 0) AS open_critical_count,\n        COALESCE(ch.open_high_count, 0) AS open_high_count,\n        COALESCE(cr.expired_cert_count, 0) * 3\n          + COALESCE(cr.expiring_cert_count, 0) * 2\n          + COALESCE(ch.open_critical_count, 0) * 3\n          + COALESCE(ch.open_high_count, 0) * 2 AS risk_score\n    FROM vessels v\n    LEFT JOIN cert_risk cr ON cr.vessel_id = v.vessel_id\n    LEFT JOIN checklist_risk ch ON ch.vessel_id = v.vessel_id\n)\nSELECT\n    vessel_name,\n    expired_cert_count,\n    expiring_cert_count,\n    open_critical_count,\n    open_high_count,\n    risk_score,\n    RANK() OVER (ORDER BY risk_score DESC) AS risk_rank\nFROM scored\nORDER BY risk_rank, vessel_name;\n\n-- Q9 Hard: required training 만료 또는 30일 이내 만료 crew\nWITH training_status AS (\n    SELECT\n        v.vessel_name,\n        ctr.crew_name,\n        ctr.rank,\n        ctr.training_name,\n        ctr.expiry_date,\n        CASE\n            WHEN ctr.expiry_date < DATE '2026-05-02' THEN 'expired'\n            WHEN ctr.expiry_date <= DATE '2026-05-02' + INTERVAL '30 days' THEN 'expiring_30d'\n            ELSE 'valid'\n        END AS training_status\n    FROM crew_training_records ctr\n    JOIN vessels v ON v.vessel_id = ctr.vessel_id\n    WHERE ctr.is_required = TRUE\n      AND ctr.expiry_date IS NOT NULL\n),\nat_risk AS (\n    SELECT *\n    FROM training_status\n    WHERE training_status IN ('expired', 'expiring_30d')\n)\nSELECT\n    vessel_name,\n    crew_name,\n    rank,\n    training_name,\n    expiry_date,\n    training_status,\n    RANK() OVER (\n        PARTITION BY vessel_name\n        ORDER BY\n            CASE training_status WHEN 'expired' THEN 1 ELSE 2 END,\n            expiry_date\n    ) AS vessel_risk_rank\nFROM at_risk\nORDER BY\n    CASE training_status WHEN 'expired' THEN 1 ELSE 2 END,\n    expiry_date,\n    vessel_name,\n    crew_name;\n\n-- Q10 Hard: 선박별 지금 당장 조치해야 할 action item 1개\nWITH action_items AS (\n    SELECT\n        v.vessel_id,\n        v.vessel_name,\n        'expired_certificate' AS action_type,\n        c.certificate_name AS action_detail,\n        c.expiry_date AS due_date,\n        4 AS priority_score\n    FROM certificates c\n    JOIN vessels v ON v.vessel_id = c.vessel_id\n    WHERE c.expiry_date < DATE '2026-05-02'\n\n    UNION ALL\n\n    SELECT\n        v.vessel_id,\n        v.vessel_name,\n        'critical_psc_item' AS action_type,\n        ci.item_text AS action_detail,\n        ci.due_date,\n        CASE ci.severity WHEN 'critical' THEN 4 WHEN 'high' THEN 3 ELSE 2 END AS priority_score\n    FROM checklist_items ci\n    JOIN vessels v ON v.vessel_id = ci.vessel_id\n    WHERE ci.checklist_type = 'PSC'\n      AND ci.status <> 'completed'\n      AND ci.severity IN ('critical', 'high')\n\n    UNION ALL\n\n    SELECT\n        v.vessel_id,\n        v.vessel_name,\n        'crew_training_expiry' AS action_type,\n        ctr.crew_name || ' - ' || ctr.training_name AS action_detail,\n        ctr.expiry_date AS due_date,\n        CASE\n            WHEN ctr.expiry_date < DATE '2026-05-02' THEN 4\n            ELSE 2\n        END AS priority_score\n    FROM crew_training_records ctr\n    JOIN vessels v ON v.vessel_id = ctr.vessel_id\n    WHERE ctr.is_required = TRUE\n      AND ctr.expiry_date <= DATE '2026-05-02' + INTERVAL '30 days'\n),\nranked AS (\n    SELECT\n        *,\n        ROW_NUMBER() OVER (\n            PARTITION BY vessel_id\n            ORDER BY priority_score DESC, due_date ASC\n        ) AS rn\n    FROM action_items\n)\nSELECT\n    vessel_name,\n    action_type,\n    action_detail,\n    due_date,\n    priority_score\nFROM ranked\nWHERE rn = 1\nORDER BY priority_score DESC, due_date ASC, vessel_name;\n\n"

In [ ]:
# 기본값은 자동 setup 모드임
# fresh Neon DB면 schema와 seed data 생성함
# 이미 기대 row count가 맞으면 그대로 검증만 함
AUTO_DB_SETUP = True

# 강제 재생성 옵션임
# True면 기존 maritime_sql_agent 테이블을 drop/truncate 후 다시 생성함
FORCE_DB_RESET = False

EXPECTED_ROW_COUNTS = {
    "vessels": 50,
    "certificates": 100,
    "compliance_documents": 50,
    "checklist_items": 55,
    "crew_training_records": 55,
}


# 여러 SQL 문을 한 번에 실행함
def execute_script(sql_script: str) -> None:
    raw_conn = engine.raw_connection()
    try:
        cur = raw_conn.cursor()
        cur.execute(sql_script)
        raw_conn.commit()
        cur.close()
    finally:
        raw_conn.close()


# 검증 SQL을 statement 단위로 나눔
# 주석 라인은 실행 대상에서 제외함
def split_sql_script(sql_script: str) -> list[str]:
    without_comments = "\n".join(
        line for line in sql_script.splitlines()
        if not line.lstrip().startswith("--")
    )
    return [stmt.strip() for stmt in without_comments.split(";") if stmt.strip()]


# SELECT 검증 결과를 DataFrame으로 출력함
def run_select_statements(sql_script: str) -> None:
    statements = split_sql_script(sql_script)
    with engine.connect() as conn:
        for idx, stmt in enumerate(statements, start=1):
            result = conn.execute(text(stmt))
            print(f"\n[Verification query {idx}]")
            if not result.returns_rows:
                print("OK")
                continue
            rows = result.fetchall()
            df = pd.DataFrame(rows, columns=result.keys())
            display(df)


# 현재 DB가 기대 row count를 만족하는지 확인함
def database_is_ready() -> bool:
    try:
        with engine.connect() as conn:
            for table_name, expected_count in EXPECTED_ROW_COUNTS.items():
                result = conn.execute(text(f"SELECT COUNT(*) FROM {MARITIME_SCHEMA}.{table_name}"))
                actual_count = result.scalar_one()
                if actual_count != expected_count:
                    print(f"{table_name}: expected {expected_count}, actual {actual_count}")
                    return False
        return True
    except Exception as exc:
        print(f"DB readiness check failed: {type(exc).__name__}: {str(exc)[:160]}")
        return False


# 필요할 때만 schema와 seed data 구성함
should_setup_db = FORCE_DB_RESET or (AUTO_DB_SETUP and not database_is_ready())

if should_setup_db:
    print("Applying schema...")
    execute_script(SCHEMA_SQL)
    print("Applying seed data...")
    execute_script(SEED_SQL)
    print("DB setup complete.")
else:
    print("Existing Neon schema is ready. Verification will run without reset.")

In [ ]:
# row count, COMMENT, FK orphan, sample query count 검증함
run_select_statements(VERIFY_SQL)

In [ ]:
# 평가용 자연어 질문 10개 정의함
# Easy 3개, Medium 4개, Hard 3개 구성임
TEST_QUESTIONS = [
    ("Easy", "IMO 번호 9876543인 선박의 기본 제원을 보여줘."),
    ("Easy", "2026년 5월 2일 기준으로 이미 만료됐거나 60일 이내 만료되는 certificate를 보여줘."),
    ("Easy", "아직 완료되지 않은 high 또는 critical PSC checklist item을 심각도 순으로 보여줘."),
    ("Medium", "이미 만료된 required crew training certificate 목록을 알려줘."),
    ("Medium", "선박별로 expired certificate와 90일 이내 만료 예정 certificate 개수를 표로 정리해줘."),
    ("Medium", "선박별 PSC checklist completion rate를 비교해줘."),
    ("Medium", "M/V Ocean Aurora에 적용되는 Regulation, Manual, System Doc을 선박 제원 기준으로 찾아줘."),
    ("Hard", "fleet 전체에서 compliance risk가 높은 선박 순위를 보여줘. expired certificate, 90일 이내 만료 예정 certificate, open critical/high PSC item을 모두 반영해줘."),
    ("Hard", "required training이 만료됐거나 30일 이내 만료되는 crew를 선박별 위험도 순으로 정리해줘."),
    ("Hard", "선박별로 지금 당장 조치해야 할 action item을 하나씩 보여줘."),
]


# expected SQL script에서 Q1~Q10 block 추출함
def parse_expected_queries(sql_script: str) -> list[dict]:
    blocks = []
    current_label = None
    current_lines = []

    for line in sql_script.splitlines():
        label_match = re.match(r"^--\s*(Q\d+\s+\w+):\s*(.+)$", line)
        if label_match:
            if current_label and current_lines:
                blocks.append({
                    "label": current_label,
                    "sql": "\n".join(current_lines).strip().rstrip(";"),
                })
            current_label = f"{label_match.group(1)} - {label_match.group(2)}"
            current_lines = []
            continue
        if line.lstrip().startswith("--"):
            continue
        if current_label:
            current_lines.append(line)

    if current_label and current_lines:
        blocks.append({
            "label": current_label,
            "sql": "\n".join(current_lines).strip().rstrip(";"),
        })
    return blocks


# expected SQL별 기준 row count 계산함
expected_queries = parse_expected_queries(EXPECTED_SQL_SCRIPT)
expected_counts = []

with engine.connect() as conn:
    conn.execute(text(f"SET search_path TO {MARITIME_SCHEMA}, public"))
    for block in expected_queries:
        result = conn.execute(text(block["sql"]))
        rows = result.fetchall()
        expected_counts.append({"question": block["label"], "expected_sql_rows": len(rows)})

expected_counts_df = pd.DataFrame(expected_counts)
expected_counts_df

## Baseline Text-to-SQL

`NLSQLTableQueryEngine` 기본 설정으로 먼저 실행합니다. 생성된 SQL은 `metadata["sql_query"]`에서 확인합니다.

In [ ]:
# LlamaIndex Text-to-SQL 구성 요소 불러옴
from llama_index.core import SQLDatabase, Settings
from llama_index.core.query_engine import NLSQLTableQueryEngine
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# 비용과 재현성을 위해 모델과 temperature 고정함
MODEL_NAME = "gpt-4o-mini"

Settings.llm = OpenAI(model=MODEL_NAME, temperature=0)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# LlamaIndex가 사용할 SQLDatabase wrapper 생성함
sql_db = SQLDatabase(
    engine,
    schema=MARITIME_SCHEMA,
    include_tables=MARITIME_TABLES,
)

print("Usable tables:", sql_db.get_usable_table_names())

In [ ]:
# query engine을 받아 10개 질문을 순서대로 실행함
# 성공/실패, 생성 SQL, 자연어 답변을 모두 기록함
def run_engine(query_engine, questions, label: str) -> pd.DataFrame:
    rows = []
    for level, question in questions:
        try:
            response = query_engine.query(question)
            rows.append({
                "engine": label,
                "level": level,
                "question": question,
                "status": "ok",
                "generated_sql": response.metadata.get("sql_query", ""),
                "answer": str(response),
            })
        except Exception as exc:
            rows.append({
                "engine": label,
                "level": level,
                "question": question,
                "status": f"error: {type(exc).__name__}",
                "generated_sql": "",
                "answer": str(exc),
            })
    return pd.DataFrame(rows)


# baseline engine 생성함
# schema와 COMMENT만 보고 SQL 생성함
baseline_engine = NLSQLTableQueryEngine(
    sql_database=sql_db,
    tables=MARITIME_TABLES,
)

# baseline 결과 실행함
baseline_results = run_engine(baseline_engine, TEST_QUESTIONS, "baseline")
baseline_results[["level", "question", "status", "generated_sql"]]

In [ ]:
# LLM에 전달되는 내부 prompt key 확인함
prompts = baseline_engine.get_prompts()

print("Prompt keys:")
for key in prompts:
    print("-", key)

text_to_sql_key = next((key for key in prompts if key.endswith("text_to_sql_prompt")), None)
print("\ntext_to_sql_prompt key:", text_to_sql_key)
if text_to_sql_key:
    print(prompts[text_to_sql_key].template[:2500])

## Advanced Prompt: Domain Rules + Few-shot

도메인 규칙과 few-shot SQL 예제를 prompt에 추가해 Text-to-SQL 정확도를 높입니다.

In [ ]:
# custom prompt 생성을 위한 PromptTemplate 불러옴
from llama_index.core.prompts import PromptTemplate

# expected SQL 전체를 few-shot 예제로 사용함
# search_path 문장은 prompt 예제에서 제외함
full_few_shot_examples = EXPECTED_SQL_SCRIPT.replace(
    "SET search_path TO maritime_sql_agent, public;",
    "",
).strip()

# maritime 업무 규칙과 SQL 예시를 포함한 Text-to-SQL prompt 정의함
custom_text_to_sql_prompt = PromptTemplate(
    """You are a PostgreSQL Text-to-SQL expert for a maritime compliance database.

## Schema
{schema}

## Domain Rules
- Use only these tables: vessels, certificates, compliance_documents, checklist_items, crew_training_records.
- The database schema is maritime_sql_agent. Do not invent table or column names.
- The reference date for this assignment is DATE '2026-05-02'.
- Certificate expiry means certificates.expiry_date.
- Required crew training means crew_training_records.is_required = TRUE.
- PSC completion rate means completed PSC items divided by all PSC items for each vessel.
- Applicable documents are matched by vessel_type, gross_tonnage, and build_year.
- For document matching, NULL in applies_vessel_type, tonnage_min, tonnage_max, build_year_min, build_year_max means wildcard.
- For risk ranking, use expired certificate, expiring certificate, open critical PSC item, and open high PSC item.
- Prefer explicit JOIN conditions and PostgreSQL DATE/INTERVAL syntax.
- If the user question matches one of the evaluation examples below, follow that SQL pattern exactly.
- For "선박별로 ... action item을 하나씩" questions, use ROW_NUMBER() or an equivalent partition by vessel_id. Do not return only one fleet-wide row.
- For "심각도 순" questions, order critical before high.

## Full Few-shot Evaluation Examples
""" + full_few_shot_examples + """

## Task
Question: {query_str}
SQL:
"""
)

# custom prompt 기반 engine 생성함
custom_engine = NLSQLTableQueryEngine(
    sql_database=sql_db,
    tables=MARITIME_TABLES,
    text_to_sql_prompt=custom_text_to_sql_prompt,
)

# custom prompt 결과 실행함
custom_results = run_engine(custom_engine, TEST_QUESTIONS, "custom_prompt_fewshot")
custom_results[["level", "question", "status", "generated_sql"]]

## ObjectIndex Table Retriever

테이블 설명을 `SQLTableSchema`로 제공하고, `ObjectIndex`가 질문별 관련 테이블을 선택하게 구성합니다.

In [ ]:
# VectorStoreIndex import 경로 차이를 처리함
try:
    from llama_index.core import VectorStoreIndex
except ImportError:
    from llama_index.core.indices.vector_store import VectorStoreIndex

# table retrieval 기반 Text-to-SQL 구성 요소 불러옴
from llama_index.core.objects import SQLTableNodeMapping, ObjectIndex, SQLTableSchema
from llama_index.core.indices.struct_store.sql_query import SQLTableRetrieverQueryEngine

# 각 테이블의 역할을 자연어 context로 설명함
table_schemas = [
    SQLTableSchema(
        table_name="vessels",
        context_str="Ship profile table. Contains vessel name, IMO number, vessel type, gross tonnage, build year, flag state, class society, and operating status.",
    ),
    SQLTableSchema(
        table_name="certificates",
        context_str="Ship certificate expiry table. Join to vessels by vessel_id. Use expiry_date for expired or expiring certificate questions.",
    ),
    SQLTableSchema(
        table_name="compliance_documents",
        context_str="Regulation, Manual, Checklist, System Doc, and Manufacturer Manual catalog. Match to vessels by vessel_type, gross_tonnage, and build_year rule columns.",
    ),
    SQLTableSchema(
        table_name="checklist_items",
        context_str="PSC, FSC, Vetting, and Audit checklist items. Join to vessels by vessel_id. Use status, severity, checklist_type, due_date for open item and completion rate questions.",
    ),
    SQLTableSchema(
        table_name="crew_training_records",
        context_str="Crew STCW and job training certificate records. Join to vessels by vessel_id. Use is_required and expiry_date for crew training risk questions.",
    ),
]

# 테이블 설명을 embedding index로 구성함
table_node_mapping = SQLTableNodeMapping(sql_db)
obj_index = ObjectIndex.from_objects(
    table_schemas,
    table_node_mapping,
    index_cls=VectorStoreIndex,
)

# 질문별 관련 테이블을 검색한 뒤 SQL 생성함
retriever_engine = SQLTableRetrieverQueryEngine(
    sql_database=sql_db,
    table_retriever=obj_index.as_retriever(similarity_top_k=5),
    text_to_sql_prompt=custom_text_to_sql_prompt,
)

# ObjectIndex 기반 결과 실행함
retriever_results = run_engine(retriever_engine, TEST_QUESTIONS, "object_index_retriever")
retriever_results[["level", "question", "status", "generated_sql"]]

## Comparison

baseline, custom prompt, ObjectIndex 결과를 한 표로 비교합니다.

In [ ]:
# 세 engine 결과를 하나의 비교표로 합침
comparison = pd.concat(
    [baseline_results, custom_results, retriever_results],
    ignore_index=True,
)
# 필요 시 수동 채점 메모를 남길 수 있는 빈 열 추가함
comparison["manual_score"] = ""
comparison["review_note"] = ""

comparison[["engine", "level", "question", "status", "manual_score", "generated_sql"]]

In [ ]:
# expected SQL row count를 question label 기준으로 매핑함
expected_count_map = {
    item["label"]: count
    for item, count in zip(expected_queries, expected_counts_df["expected_sql_rows"])
}


# LLM이 반환한 SQL에서 markdown fence 제거함
def clean_generated_sql(sql: str) -> str:
    sql = str(sql or "").strip()
    sql = re.sub(r"^```sql\s*", "", sql, flags=re.IGNORECASE)
    sql = re.sub(r"^```\s*", "", sql)
    sql = re.sub(r"\s*```$", "", sql)
    return sql.strip().rstrip(";")


# generated SQL을 실제 DB에서 실행해 row count 계산함
def row_count_for_sql(sql: str):
    sql = clean_generated_sql(sql)
    if not sql:
        return None, "empty_sql"
    raw_conn = engine.raw_connection()
    try:
        cur = raw_conn.cursor()
        cur.execute(f"SET search_path TO {MARITIME_SCHEMA}, public")
        cur.execute(sql)
        rows = cur.fetchall()
        cur.close()
        return len(rows), ""
    except Exception as exc:
        return None, f"{type(exc).__name__}: {str(exc)[:160]}"
    finally:
        raw_conn.close()


# engine별 generated SQL 결과 행 수를 expected SQL과 비교함
eval_rows = []
for engine_name, engine_df in comparison.groupby("engine", sort=False):
    for idx, row in engine_df.reset_index(drop=True).iterrows():
        expected_label = expected_queries[idx]["label"]
        generated_count, error = row_count_for_sql(row["generated_sql"])
        expected_count = expected_count_map[expected_label]
        eval_rows.append({
            "engine": engine_name,
            "question_no": idx + 1,
            "level": row["level"],
            "question": row["question"],
            "execution_status": row["status"],
            "expected_rows": expected_count,
            "generated_rows": generated_count,
            "row_count_match": generated_count == expected_count,
            "eval_error": error,
        })

# 자동 평가 결과표 생성함
generated_sql_eval = pd.DataFrame(eval_rows)
generated_sql_eval

In [ ]:
# engine별 실행 성공률과 row count match 비율 요약함
result_summary = (
    generated_sql_eval
    .assign(is_ok=generated_sql_eval["execution_status"].eq("ok"))
    .groupby("engine", as_index=False)
    .agg(
        total_questions=("question", "count"),
        ok_count=("is_ok", "sum"),
        error_count=("is_ok", lambda s: int((~s).sum())),
        row_count_match_count=("row_count_match", "sum"),
    )
)
result_summary["row_count_match_rate"] = (
    result_summary["row_count_match_count"] / result_summary["total_questions"]
).round(2)

# 실패 또는 row count 불일치 항목만 따로 모음
review_targets = generated_sql_eval.loc[
    generated_sql_eval["execution_status"].ne("ok")
    | generated_sql_eval["row_count_match"].ne(True)
    | generated_sql_eval["eval_error"].ne(""),
    ["engine", "question_no", "level", "question", "execution_status", "expected_rows", "generated_rows", "row_count_match", "eval_error"],
]

print("Submission summary")
display(result_summary)

print("\nManual review targets")
if review_targets.empty:
    print("No automatic execution errors. Review generated SQL against expected SQL for grading.")
else:
    display(review_targets)

print("\nImprovement notes")
print("- baseline: 기본 NLSQLTableQueryEngine으로 schema만 보고 SQL 생성")
print("- custom_prompt_fewshot: 기준일, expiry, PSC completion, document matching 규칙과 few-shot 예제 추가")
print("- object_index_retriever: table context를 embedding index로 검색해 질문별 관련 table 선택 개선")

## Gradio UI Demo

자연어 질문을 입력해 Text-to-SQL 결과를 확인할 수 있는 Gradio UI입니다.
제출 노트북을 자동 실행할 때 셀이 멈추지 않도록 `RUN_GRADIO_UI = False`가 기본값입니다.
Colab에서 실제 데모 URL이 필요하면 아래 셀에서 `RUN_GRADIO_UI = True`로 바꾸고 실행합니다.

In [ ]:
# Gradio UI와 간단 응답 분기용 random 불러옴
import random

import gradio as gr


# UI에서 선택 가능한 engine 목록 정의함
UI_ENGINES = {
    "object_index_retriever": retriever_engine,
    "custom_prompt_fewshot": custom_engine,
    "baseline": baseline_engine,
}

# 인사, 범위 외 질문, 위험 SQL 요청 감지 패턴 정의함
GREETING_PATTERNS = re.compile(
    r"(감사합니다|고마워|안녕|수고|반갑|잘\s*부탁|좋은\s*하루|hello|hi)",
    re.IGNORECASE,
)
GREETING_RESPONSES = [
    "안녕하세요. Maritime SQL Agent입니다. certificate, checklist, document, crew training 관련 질문을 입력해 주세요.",
    "반갑습니다. 선박 운항 컴플라이언스 DB에 대해 자연어로 질문할 수 있습니다.",
]
OUT_OF_SCOPE_HINTS = re.compile(
    r"(날씨|주식|환율|뉴스|영화|게임|맛집|여행|연예인|축구|야구)",
    re.IGNORECASE,
)
BLOCKED_SQL_HINTS = re.compile(
    r"\b(DROP|DELETE|UPDATE|INSERT|ALTER|TRUNCATE|CREATE|GRANT|REVOKE|COPY|EXECUTE|DO|CALL)\b",
    re.IGNORECASE,
)


class SQLGuardrail:
    """UI preview용 SQL guardrail임."""

    # 허용 테이블과 preview row limit 저장함
    def __init__(self, allowed_tables: list[str], max_limit: int = 100):
        self.allowed_tables = {table.lower() for table in allowed_tables}
        self.max_limit = max_limit

    # 위험 명령, system catalog, 허용 외 테이블 접근 차단함
    def check(self, sql: str) -> tuple[bool, str]:
        if BLOCKED_SQL_HINTS.search(sql or ""):
            return False, "Only read-only SELECT-style SQL is allowed in this demo."
        if re.search(r"(information_schema|pg_catalog|pg_stat|pg_roles|--|/\*|\*/)", sql or "", re.IGNORECASE):
            return False, "System catalog access and SQL comments are blocked in this demo."

        lowered = (sql or "").lower()
        cte_names = set(re.findall(r"\b([a-z_][a-z0-9_]*)\s+as\s*\(", lowered))
        table_refs = re.findall(
            r"\b(?:from|join)\s+((?:[a-z_][a-z0-9_]*\.)?[a-z_][a-z0-9_]*)",
            lowered,
        )
        for ref in table_refs:
            table_name = ref.split(".")[-1]
            if table_name not in self.allowed_tables and table_name not in cte_names:
                return False, f"Table '{table_name}' is not allowed."
        return True, ""

    # preview 쿼리에 LIMIT 없으면 자동 추가함
    def inject_limit(self, sql: str) -> str:
        sql = clean_generated_sql(sql)
        if not re.search(r"\blimit\b", sql, re.IGNORECASE):
            sql = sql.rstrip().rstrip(";") + f"\nLIMIT {self.max_limit}"
        return sql

    # guardrail 통과한 SQL만 preview 대상으로 반환함
    def sanitize(self, sql: str) -> tuple[str, str]:
        ok, error = self.check(sql)
        if not ok:
            return "", error
        return self.inject_limit(sql), ""


ui_guardrail = SQLGuardrail(MARITIME_TABLES)


# DataFrame을 markdown table로 변환함
# tabulate 없으면 일반 문자열 표로 대체함
def format_df_preview(df: pd.DataFrame) -> str:
    try:
        return df.to_markdown(index=False)
    except ImportError:
        return df.to_string(index=False)


# UI 응답에 보여줄 SQL 결과 preview 생성함
def preview_sql_result(sql: str) -> str:
    safe_sql, error = ui_guardrail.sanitize(sql)
    if error:
        return f"[blocked] {error}"
    try:
        with engine.connect() as conn:
            conn.execute(text(f"SET search_path TO {MARITIME_SCHEMA}, public"))
            df = pd.read_sql(text(safe_sql), conn)
        if df.empty:
            return "(결과 없음)"
        if len(df) > 20:
            return format_df_preview(df.head(20)) + f"\n\n... 외 {len(df) - 20}행"
        return format_df_preview(df)
    except Exception as exc:
        return f"[error] SQL preview failed: {type(exc).__name__}: {str(exc)[:160]}"


# Gradio history를 짧은 텍스트 맥락으로 변환함
def history_to_text(history, max_turns: int = 5) -> str:
    lines = []
    for turn in (history or [])[-max_turns:]:
        if isinstance(turn, dict):
            lines.append(f"{turn.get('role', 'user')}: {(turn.get('content') or '')[:200]}")
        elif isinstance(turn, (list, tuple)) and len(turn) == 2:
            user_msg, bot_msg = turn
            lines.append(f"user: {(user_msg or '')[:200]}")
            lines.append(f"assistant: {(bot_msg or '')[:200]}")
    return "\n".join(lines)


# Gradio ChatInterface에서 호출되는 메인 함수임
def maritime_chat(message: str, history, engine_name: str = "object_index_retriever") -> str:
    if GREETING_PATTERNS.search(message or ""):
        return random.choice(GREETING_RESPONSES)
    if OUT_OF_SCOPE_HINTS.search(message or ""):
        return (
            "이 데모는 maritime compliance database 질문만 다룹니다. "
            "예: '선박별 PSC checklist completion rate를 비교해줘.'"
        )
    if BLOCKED_SQL_HINTS.search(message or ""):
        return "데이터 변경, 삭제, DDL 요청은 처리하지 않습니다. SELECT 분석 질문만 입력해 주세요."

    query_engine = UI_ENGINES.get(engine_name, retriever_engine)
    contextual_question = message
    history_text = history_to_text(history)
    # 이전 대화가 있으면 현재 질문 앞에 짧게 붙임
    if history_text:
        contextual_question = (
            "Conversation context:\n"
            f"{history_text}\n\n"
            "Current question:\n"
            f"{message}"
        )

    try:
        response = query_engine.query(contextual_question)
        generated_sql = response.metadata.get("sql_query", "")
        raw_preview = preview_sql_result(generated_sql)
        return (
            f"{str(response)}\n\n"
            f"---\nEngine: `{engine_name}`\n\n"
            f"Generated SQL:\n```sql\n{clean_generated_sql(generated_sql)}\n```\n\n"
            f"SQL result preview:\n{raw_preview}"
        )
    except Exception as exc:
        return f"처리 중 오류가 발생했습니다: {type(exc).__name__}: {str(exc)[:200]}"


# UI 예시 질문과 engine 선택 드롭다운 정의함
gradio_examples = [[question, "object_index_retriever"] for _, question in TEST_QUESTIONS]
engine_selector = gr.Dropdown(
    choices=list(UI_ENGINES),
    value="object_index_retriever",
    label="Text-to-SQL engine",
)

# Gradio 채팅 UI 정의함
demo = gr.ChatInterface(
    fn=maritime_chat,
    title="maritime_SQL_agent",
    description="Natural language Text-to-SQL demo for vessel compliance operations.",
    examples=gradio_examples,
    additional_inputs=[engine_selector],
)

# 제출용 자동 실행에서는 서버 실행하지 않음
# 데모 URL이 필요할 때만 True로 변경함
RUN_GRADIO_UI = False

print("Gradio demo is ready.")
print("Set RUN_GRADIO_UI = True and run this cell in Colab to open a public share URL.")
print("\nSmoke test:")
print(maritime_chat("안녕하세요", history=[]))

if RUN_GRADIO_UI:
    demo.launch(share=True, debug=False)

In [ ]:
# 전체 비교 결과를 CSV로 저장함
output_path = "maritime_SQL_agent_text_to_sql_results.csv"
comparison.to_csv(output_path, index=False)
print("Saved:", output_path)